In [1]:
# ==========================================================
# Notebook 2
# Generate TFRecords
# ==========================================================

# Install CESNET DataZoo (safe if already installed)
!pip -q install cesnet-datazoo==0.2.0
import gc
import json
import shutil
import warnings
import logging
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler

from cesnet_datazoo.datasets import CESNET_QUICEXT_25
from cesnet_datazoo.config import DatasetConfig, AppSelection, ValidationApproach

warnings.filterwarnings("ignore")

print("===== Notebook 2 =====")
print(f"TensorFlow : {tf.__version__}")
print(f"Numpy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("======================")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 7.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
===== Notebook 2 =====
TensorFlow : 2.20.0
Numpy      : 2.0.2
Pandas     : 2.3.3


In [2]:
# ==========================================================
# Configuration
# ==========================================================

from pathlib import Path

print("Loading configuration...")

# ----------------------------------------------------------
# Input dataset (read-only Kaggle Dataset)
# ----------------------------------------------------------

INPUT_DATASET = Path(
    "/kaggle/input/datasets/yahiaa1/cesnet-quixext-25-m/CESNET-QUICEXT-25"
)

# ----------------------------------------------------------
# Writable copy (required by cesnet-datazoo)
# ----------------------------------------------------------

WORKING_DATASET = Path(
    "/kaggle/working/CESNET-QUICEXT-25"
)

DB_NAME = "CESNET-QUICEXT-25-M.h5"
src_m = INPUT_DATASET / "M"
dst_m = WORKING_DATASET / "M"

assert (src_m / DB_NAME).exists(), "Attach the CESNET-QUICEXT-25-M dataset under Add Input."

def working_copy_is_valid() -> bool:
    db = dst_m / DB_NAME
    return (
        db.exists()
        and db.stat().st_size == (src_m / DB_NAME).stat().st_size  # rejects truncated copies (stat follows symlinks)
        and (dst_m / "servicemap.csv").exists()
    )

if not working_copy_is_valid():
    print("Working dataset missing or incomplete -> rebuilding...")
    shutil.rmtree(WORKING_DATASET, ignore_errors=True)
    dst_m.mkdir(parents=True)
    (dst_m / DB_NAME).symlink_to(src_m / DB_NAME)      # 17.66 GB database: link, don't copy
    shutil.copy(src_m / "servicemap.csv", dst_m / "servicemap.csv")
    assert working_copy_is_valid()
    print("Working dataset ready.")
else:
    print("Writable dataset already valid.")

DATASET_DIR = WORKING_DATASET


# --- Place metadata files created in notebook 1
tfrecord_dataset_dir = Path('/kaggle/working/tfrecord_dataset')
if not tfrecord_dataset_dir.exists():
    print('Creating /kaggle/working/tfrecord_dataset')
    os.makedirs(tfrecord_dataset_dir, exist_ok=True)
artifacts_dir = Path('/kaggle/working/tfrecord_dataset/artifacts')
metadata_dir = Path('/kaggle/working/tfrecord_dataset/metadata')

source_dir = '/kaggle/input/datasets/yahiaa1/metadata-files-202406-202503'
if not artifacts_dir.exists():
    print('Creating /kaggle/working/tfrecord_dataset/artifacts')
    os.makedirs(artifacts_dir, exist_ok=True)
    print('Copying class_weights_202406_202503.npy')
    shutil.copy(os.path.join(source_dir, 'class_weights_202406_202503.npy'), os.path.join(artifacts_dir, 'class_weights_202406_202503.npy'))
    print('Copying preprocessing_202406_202503.joblib')
    shutil.copy(os.path.join(source_dir, 'preprocessing_202406_202503.joblib'), os.path.join(artifacts_dir, 'preprocessing_202406_202503.joblib'))

    os.makedirs(artifacts_dir, exist_ok=True)
if not metadata_dir.exists():
    print('Creating /kaggle/working/tfrecord_dataset/metadata')
    os.makedirs(metadata_dir, exist_ok=True)
    print('Copying dataset_metadata_202406_202503.json')
    shutil.copy(os.path.join(source_dir, 'dataset_metadata_202406_202503.json'), os.path.join(metadata_dir, 'dataset_metadata_202406_202503.json'))


print("Metadata files successfully moved into position!")

# ----------------------------------------------------------
# Output directory
# ----------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/tfrecord_dataset"
)

TFRECORD_DIR = OUTPUT_DIR / "tfrecords"

TFRECORD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ----------------------------------------------------------
# Artifact locations
# ----------------------------------------------------------

ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
METADATA_DIR = OUTPUT_DIR / "metadata"

PREPROCESSING_FILE = ARTIFACT_DIR / "preprocessing_202406_202503.joblib"
CLASS_WEIGHTS_FILE = ARTIFACT_DIR / "class_weights_202406_202503.npy"
DATASET_METADATA_FILE = METADATA_DIR / "dataset_metadata_202406_202503.json"

# ----------------------------------------------------------
# Load preprocessing artifacts
# ----------------------------------------------------------

preprocessing = joblib.load(
    PREPROCESSING_FILE
)

iat_scaler = preprocessing["iat_scaler"]
size_scaler = preprocessing["size_scaler"]
aux_scaler = preprocessing["aux_scaler"]
aux_columns = preprocessing["aux_columns"]

class_weights = np.load(
    CLASS_WEIGHTS_FILE
)

with open(DATASET_METADATA_FILE, "r") as f:
    metadata = json.load(f)

# ----------------------------------------------------------
# Recover configuration
# ----------------------------------------------------------

SIZE = metadata["dataset_size"]

TARGET_COLUMN = metadata["target_column"]

TRAIN_MONTHS = metadata["train_months"]
VAL_MONTH = metadata["val_month"]
TEST_PERIOD_NAME = metadata["test_period"]

TRAIN_SHARDS = metadata["train_shards"]
VAL_SHARDS = metadata["validation_shards"]
TEST_SHARDS = metadata["test_shards"]

NUM_CLASSES = metadata["num_classes"]

NUM_AUX_FEATURES = metadata["num_aux_features"]

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()

print("Dataset :", DATASET_DIR)
print("Output  :", OUTPUT_DIR)

print()

print("Training months :", TRAIN_MONTHS)
print("Validation month:", VAL_MONTH)
print("Classes         :", NUM_CLASSES)
print("Aux features    :", NUM_AUX_FEATURES)

print()

print("Configuration loaded.")

Loading configuration...
Working dataset missing or incomplete -> rebuilding...
Working dataset ready.
Creating /kaggle/working/tfrecord_dataset
Creating /kaggle/working/tfrecord_dataset/artifacts
Copying class_weights_202406_202503.npy
Copying preprocessing_202406_202503.joblib
Creating /kaggle/working/tfrecord_dataset/metadata
Copying dataset_metadata_202406_202503.json
Metadata files successfully moved into position!

Dataset : /kaggle/working/CESNET-QUICEXT-25
Output  : /kaggle/working/tfrecord_dataset

Training months : ['202406', '202407', '202408', '202409', '202410', '202411', '202412', '202501', '202502', '202503']
Validation month: 202504
Classes         : 50
Aux features    : 11

Configuration loaded.


In [3]:
# ==========================================================
# Utility functions
# ==========================================================

print("Loading utilities...")

# ----------------------------------------------------------
# JSON
# ----------------------------------------------------------

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w") as f:
        json.dump(obj, f, indent=4)


# ----------------------------------------------------------
# TFRecord feature helpers
# ----------------------------------------------------------

def bytes_feature(value: bytes):
    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=[value])
    )


def int64_feature(value: int):
    return tf.train.Feature(
        int64_list=tf.train.Int64List(value=[int(value)])
    )


# ----------------------------------------------------------
# Serialize one example
# ----------------------------------------------------------

def serialize_example(ppi, aux, label):

    feature = {
        "ppi": bytes_feature(
            ppi.astype(np.float32).tobytes()
        ),
        "label": int64_feature(label),
    }

    if aux is not None:
        feature["aux"] = bytes_feature(
            aux.astype(np.float32).tobytes()
        )

    example = tf.train.Example(
        features=tf.train.Features(feature=feature)
    )

    return example.SerializeToString()


# ----------------------------------------------------------
# PPI preprocessing
# ----------------------------------------------------------

def preprocess_ppi(ppi):

    # (N,3,30)
    ppi = np.stack(
        ppi.map(
            lambda x: np.stack(x).astype(np.float32)
        ).to_numpy()
    )
    # Padding mask (direction==0 marks a padded packet slot). Must be
    # captured BEFORE scaling — StandardScaler maps raw 0 to a non-zero
    # value, which otherwise destroys the "no packet here" signal for
    # every flow shorter than 30 packets.
    padding_mask = ppi[:, 1, :] == 0
    
    # Channel 0
    ppi[:, 0, :] = np.log1p(ppi[:, 0, :])
    ppi[:, 0, :] = iat_scaler.transform(ppi[:, 0, :])
    ppi[:, 0, :][padding_mask] = 0.0
    # Channel 2
    ppi[:, 2, :] = np.log1p(ppi[:, 2, :])
    ppi[:, 2, :] = size_scaler.transform(ppi[:, 2, :])
    ppi[:, 2, :][padding_mask] = 0.0
    # (N,30,3)
    ppi = np.transpose(
        ppi,
        (0, 2, 1),
    )

    return ppi.astype(np.float32)


# ----------------------------------------------------------
# Auxiliary preprocessing
# ----------------------------------------------------------

def preprocess_aux(df):

    if len(aux_columns) == 0:
        return None

    aux = df[aux_columns].to_numpy(
        dtype=np.float32
    )

    np.clip(
        aux,
        0,
        None,
        out=aux,
    )

    aux = np.log1p(aux)

    aux = aux_scaler.transform(aux)

    return aux.astype(np.float32)


print("Utilities loaded.")

Loading utilities...
Utilities loaded.


In [4]:
# ==========================================================
# Dataset loading helpers
# ==========================================================

print("Loading dataset helpers...")

dataset = CESNET_QUICEXT_25(
    str(DATASET_DIR),
    size=SIZE,
)

AVAILABLE_DATES = dataset.available_dates


def get_month_dates(month: str):
    """
    Return every available date belonging to one month.
    Example:
        202501 ->
        ["20250101", "20250102", ..., "20250131"]
    """
    return [
        d
        for d in AVAILABLE_DATES
        if d.startswith(month)
    ]


def load_training_month(month: str) -> pd.DataFrame:
    """
    Load one training month as a DataFrame.
    """

    month_dates = get_month_dates(month)

    if len(month_dates) == 0:
        raise RuntimeError(
            f"No dates found for {month}"
        )

    config = DatasetConfig(
        dataset=dataset,
        apps_selection=AppSelection.ALL_KNOWN,

        train_period_name=f"month_{month}",
        train_dates=month_dates,

        need_val_set=False,
        need_test_set=False,

        train_workers=2,
        batch_size=128
    )

    dataset.set_dataset_config_and_initialize(
        config
    )
    df = dataset.get_train_df()

    return df


# def load_validation_and_test():
#     """
#     Load validation and test exactly once.
#     """

#     all_train_dates = []

#     for month in TRAIN_MONTHS:
#         all_train_dates.extend(
#             get_month_dates(month)
#         )

#     config = DatasetConfig(
#         dataset=dataset,
#         apps_selection=AppSelection.ALL_KNOWN,

#         train_period_name="all_training",
#         train_dates=all_train_dates,

#         test_period_name=TEST_PERIOD_NAME,

#         val_workers=1,
#         test_workers=1,
#         test_batch_size=1024
#     )

#     dataset.set_dataset_config_and_initialize(
#         config
#     )
#     val_df = dataset.get_val_df()
#     test_df = dataset.get_test_df()

#     return val_df, test_df
def load_validation_and_test():
    """
    Load validation (a genuine temporal holdout month, VAL_MONTH) and
    test exactly once. train_dates/train_period_name are still required
    by DatasetConfig (need_train_set defaults to True and can't be
    disabled under AppSelection.ALL_KNOWN), but the resulting train
    split from this call is never used -- training shards are written
    separately, per month, in the next cell.
    """

    all_train_dates = []
    for month in TRAIN_MONTHS:
        all_train_dates.extend(get_month_dates(month))
    
    val_dates = get_month_dates(VAL_MONTH)

    config = DatasetConfig(
        dataset=dataset,
        apps_selection=AppSelection.ALL_KNOWN,

        train_period_name="all_training",
        train_dates=all_train_dates,

        val_approach=ValidationApproach.VALIDATION_DATES,
        val_period_name=f"month_{VAL_MONTH}",
        val_dates=val_dates,

        test_period_name=TEST_PERIOD_NAME,

        val_workers=1,
        test_workers=1,
        test_batch_size=1024,
    )

    dataset.set_dataset_config_and_initialize(
          config
    )

    val_df = dataset.get_val_df()
    test_df = dataset.get_test_df()

    return val_df, test_df
print("Dataset helpers loaded.")

Loading dataset helpers...
Dataset helpers loaded.


In [14]:
# ==========================================================
# Write training TFRecords (ONE MONTH)
# ==========================================================
!rm -rf /kaggle/working/tfrecord_dataset/tfrecords/*
CURRENT_MONTH = "202503"

print("Writing training TFRecords...")
print(f"Current month : {CURRENT_MONTH}")

# ----------------------------------------------------------
# Shard assignment (Assign more shards to lengthy months)
# ----------------------------------------------------------

TRAIN_SHARD_MAP = {
    "202406": (0, 7),
    "202407": (7, 6),
    "202408": (13, 6),
    "202409": (19, 10),
    "202410": (29, 12),
    "202411": (41, 13),
    "202412": (54, 7),
    "202501": (61, 11),
    "202502": (72, 8),
    "202503": (80, 7),
}

assert sum(
    num_shards
    for _, num_shards in TRAIN_SHARD_MAP.values()
) == 87

assert CURRENT_MONTH in TRAIN_SHARD_MAP

FIRST_SHARD, N_SHARDS = TRAIN_SHARD_MAP[CURRENT_MONTH]

# ----------------------------------------------------------
# Load month
# ----------------------------------------------------------

print()
print(f"Loading {CURRENT_MONTH}...")

df = load_training_month(CURRENT_MONTH)

print(f"Rows : {len(df):,}")

# ----------------------------------------------------------
# Preprocess
# ----------------------------------------------------------

X_ppi = preprocess_ppi(df["PPI"])
X_aux = preprocess_aux(df)

y = df[TARGET_COLUMN].to_numpy(
    dtype=np.int64
)

del df
gc.collect()

# ----------------------------------------------------------
# Write shards
# ----------------------------------------------------------

n_samples = len(y)

shard_size = (n_samples + N_SHARDS - 1) // N_SHARDS

print(f"Writing {N_SHARDS} shard(s)...")

generated_files = []

for local_shard in range(N_SHARDS):

    start = local_shard * shard_size
    end = min(start + shard_size, n_samples)

    global_shard = FIRST_SHARD + local_shard

    output_file = (
        TFRECORD_DIR /
        f"train_{global_shard:03d}.tfrecord"
    )

    generated_files.append(output_file)

    with tf.io.TFRecordWriter(str(output_file)) as writer:

        for i in range(start, end):

            writer.write(
                serialize_example(
                    X_ppi[i],
                    None if X_aux is None else X_aux[i],
                    y[i],
                )
            )

    print(
        f"train_{global_shard:03d}.tfrecord "
        f"({end-start:,} examples)"
    )

# ----------------------------------------------------------
# Cleanup
# ----------------------------------------------------------

del X_ppi
del X_aux
del y

gc.collect()

print()
print("Month completed.")
print()

print("Generated files:")

for f in generated_files:
    print(" ", f.name)

print()
print("NEXT STEPS")
print("==========")
print("1. Download these TFRecord files to your local machine.")
print("2. Verify the download completed successfully.")
print("3. Delete the downloaded TFRecords from /kaggle/working.")
print("4. Change CURRENT_MONTH to the next month.")
print("5. Run this cell again.")

print()
print("To delete only this month's TFRecords after downloading:")

for f in generated_files:
    print(f"!rm '{f}'")

Writing training TFRecords...
Current month : 202503

Loading 202503...
Loading data from dataloader


100%|██████████| 17861/17861 [00:49<00:00, 358.32it/s]


Rows : 2,286,199
Writing 7 shard(s)...
train_080.tfrecord (326,600 examples)
train_081.tfrecord (326,600 examples)
train_082.tfrecord (326,600 examples)
train_083.tfrecord (326,600 examples)
train_084.tfrecord (326,600 examples)
train_085.tfrecord (326,600 examples)
train_086.tfrecord (326,599 examples)

Month completed.

Generated files:
  train_080.tfrecord
  train_081.tfrecord
  train_082.tfrecord
  train_083.tfrecord
  train_084.tfrecord
  train_085.tfrecord
  train_086.tfrecord

NEXT STEPS
1. Download these TFRecord files to your local machine.
2. Verify the download completed successfully.
3. Delete the downloaded TFRecords from /kaggle/working.
4. Change CURRENT_MONTH to the next month.
5. Run this cell again.

To delete only this month's TFRecords after downloading:
!rm '/kaggle/working/tfrecord_dataset/tfrecords/train_080.tfrecord'
!rm '/kaggle/working/tfrecord_dataset/tfrecords/train_081.tfrecord'
!rm '/kaggle/working/tfrecord_dataset/tfrecords/train_082.tfrecord'
!rm '/kaggl

In [15]:
!rm -rf /kaggle/working/tfrecord_dataset/tfrecords/*

In [16]:
# ==========================================================
# Write validation and test TFRecords
# ==========================================================

print("Writing validation and test TFRecords...")

# ----------------------------------------------------------
# Load validation and test
# ----------------------------------------------------------

val_df, test_df = load_validation_and_test()
print(f"Validation rows : {len(val_df):,}")
print(f"Test rows       : {len(test_df):,}")


def write_split(df, prefix, n_shards):

    X_ppi = preprocess_ppi(df["PPI"])
    X_aux = preprocess_aux(df)

    y = df[TARGET_COLUMN].to_numpy(
        dtype=np.int64
    )

    n_samples = len(y)
    shard_size = (n_samples + n_shards - 1) // n_shards

    print(f"\nWriting {prefix}...")

    for shard in range(n_shards):

        start = shard * shard_size
        end = min(start + shard_size, n_samples)

        path = TFRECORD_DIR / f"{prefix}_{shard:03d}.tfrecord"

        with tf.io.TFRecordWriter(str(path)) as writer:

            for i in range(start, end):

                writer.write(
                    serialize_example(
                        X_ppi[i],
                        None if X_aux is None else X_aux[i],
                        y[i],
                    )
                )

        print(
            f"{prefix}_{shard:03d}.tfrecord "
            f"({end-start:,} examples)"
        )

    del X_ppi
    del X_aux
    del y
    gc.collect()


Writing validation and test TFRecords...
Loading data from dataloader


100%|██████████| 2515/2515 [00:38<00:00, 65.11it/s]


Loading data from dataloader


100%|██████████| 2304/2304 [00:33<00:00, 69.04it/s]


Validation rows : 2,575,093
Test rows       : 2,359,061


In [17]:
# free some space for validation and test tfrecords
!rm -rf /kaggle/working/CESNET-QUICEXT-25

# ----------------------------------------------------------
# Validation
# ----------------------------------------------------------

write_split(
    val_df,
    "val",
    VAL_SHARDS,
)

del val_df
gc.collect()

# ----------------------------------------------------------
# Test
# ----------------------------------------------------------

write_split(
    test_df,
    "test",
    TEST_SHARDS,
)

del test_df
gc.collect()

print()
print("Validation TFRecords completed.")
print("Test TFRecords completed.")


Writing val...
val_000.tfrecord (321,887 examples)
val_001.tfrecord (321,887 examples)
val_002.tfrecord (321,887 examples)
val_003.tfrecord (321,887 examples)
val_004.tfrecord (321,887 examples)
val_005.tfrecord (321,887 examples)
val_006.tfrecord (321,887 examples)
val_007.tfrecord (321,884 examples)

Writing test...
test_000.tfrecord (294,883 examples)
test_001.tfrecord (294,883 examples)
test_002.tfrecord (294,883 examples)
test_003.tfrecord (294,883 examples)
test_004.tfrecord (294,883 examples)
test_005.tfrecord (294,883 examples)
test_006.tfrecord (294,883 examples)
test_007.tfrecord (294,880 examples)

Validation TFRecords completed.
Test TFRecords completed.


In [19]:
# ==========================================================
# Verify generated dataset
# ==========================================================

print("Verifying generated dataset...")

from pathlib import Path

# ----------------------------------------------------------
# Collect TFRecord files
# ----------------------------------------------------------

# train files were deleted previously
# train_files = sorted(
#     TFRECORD_DIR.glob("train_*.tfrecord")
# )

val_files = sorted(
    TFRECORD_DIR.glob("val_*.tfrecord")
)

test_files = sorted(
    TFRECORD_DIR.glob("test_*.tfrecord")
)

# ----------------------------------------------------------
# File counts
# ----------------------------------------------------------

print()

# print(f"Train shards : {len(train_files)}")
print(f"Val shards   : {len(val_files)}")
print(f"Test shards  : {len(test_files)}")

# assert len(train_files) == TRAIN_SHARDS
assert len(val_files) == VAL_SHARDS
assert len(test_files) == TEST_SHARDS

# ----------------------------------------------------------
# Total size
# ----------------------------------------------------------

all_files = val_files + test_files #+ train_files

total_size = sum(
    f.stat().st_size
    for f in all_files
)

print()
print(f"Total TFRecord size : {total_size / 1024**3:.2f} GB")

# ----------------------------------------------------------
# Metadata sanity check
# ----------------------------------------------------------

print()
print("Metadata")

print(f"Classes      : {NUM_CLASSES}")
print(f"PPI shape    : {metadata['ppi_shape']}")
print(f"Aux features : {NUM_AUX_FEATURES}")

# ----------------------------------------------------------
# Verify one example
# ----------------------------------------------------------

feature_description = {
    "ppi": tf.io.FixedLenFeature([], tf.string),
    "aux": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

raw_dataset = tf.data.TFRecordDataset(
    [str(val_files[0])]
)

example = next(iter(raw_dataset))

parsed = tf.io.parse_single_example(
    example,
    feature_description,
)

ppi = np.frombuffer(
    parsed["ppi"].numpy(),
    dtype=np.float32,
).reshape(30, 3)

aux = np.frombuffer(
    parsed["aux"].numpy(),
    dtype=np.float32,
)

label = int(
    parsed["label"].numpy()
)

print()
print("Example verification")

print("PPI shape :", ppi.shape)
print("Aux shape :", aux.shape)
print("Label     :", label)

assert ppi.shape == (30, 3)
assert aux.shape[0] == NUM_AUX_FEATURES
assert 0 <= label < NUM_CLASSES

print()
print("Notebook 2 completed successfully.")

Verifying generated dataset...

Val shards   : 8
Test shards  : 8

Total TFRecord size : 2.16 GB

Metadata
Classes      : 50
PPI shape    : [30, 3]
Aux features : 11


2026-09-28 17:50:10.688401: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)



Example verification
PPI shape : (30, 3)
Aux shape : (11,)
Label     : 4

Notebook 2 completed successfully.
